# UPI Payment Intelligence Analysis

## Project Objective
A fintech company wants to understand fraud patterns, merchant performance, 
and user behaviour across India's UPI payment ecosystem. As a Data Analyst, 
the goal is to identify high-risk segments, top performing merchant categories, 
and key drivers of transaction failures — and provide actionable recommendations 
for fraud prevention and business expansion.

## Business Questions We Are Solving
1. What is the fraud profile — which city tier, device, app, and user segment has the highest fraud rate?
2. Which merchant categories drive the highest transaction volume and value?
3. How does spending behaviour differ across age groups and city tiers?
4. What drives transaction failures — and what is the operational cost?
5. What is the risk profile of high value transactions and what should the fraud prevention strategy be?

## Tools Used
- Python + Pandas — Data ingestion and cleaning
- PostgreSQL — Business analysis queries
- Excel — Interactive dashboard
- Matplotlib + Seaborn — Visualisations

## Phase 1 — Data Ingestion & Initial Inspection

### What is Data Ingestion?
Data Ingestion is the process of loading raw data from its source into 
our working environment. In real fintech companies, data analysts receive 
data from multiple sources — databases, APIs, CSV exports. Here we are 
loading 4 relational tables that together form a complete UPI transaction 
ecosystem.

### What is Initial Inspection?
Before touching any data we need to understand what we are working with. 
This means checking the shape, structure, data types, and quality of each 
table. Skipping this step is how analysts make mistakes in real projects.

### Why 4 Separate Tables?
In real fintech systems data is never stored in one flat file. It lives 
in a relational database — transactions in one table, users in another, 
merchants in another. This is exactly how PostgreSQL databases work in 
production. We are replicating that real world structure here.

In [1]:
#---------------------------------------------------------------
# PHASE 1 — DATA INGESTION & INITIAL INSPECTION
#---------------------------------------------------------------

# Step 1 - Lets impart all libraries 
import pandas as pd   # For Data manipulation and analysis
import numpy as np    # For numerical operations 
import matplotlib.pyplot as plt  # for visualization 
import seaborn as sns # for statistical visualization 
import os             # for file path handling 


import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
# Step 2 - Load all csv files into saverate Dataframes

transactions = pd.read_csv('../data/raw/transactions.csv')
users = pd.read_csv('../data/raw/users.csv')
merchants = pd.read_csv('../data/raw/merchants.csv')
fraud_labels = pd.read_csv('../data/raw/fraud_labels.csv')


# Create backups of raw data — never modify original
transactions_raw = transactions.copy()
users_raw = users.copy()
merchants_raw = merchants.copy()
fraud_labels_raw = fraud_labels.copy()


print(" All 4 datasets loaded successfully")


 All 4 datasets loaded successfully


In [3]:
# Step 3 — Check shape of all 4 tables

print('transactions:', transactions.shape)
print('users:', users.shape)
print('merchants', merchants.shape)
print('fraud_labels:', fraud_labels.shape)


transactions: (20000, 30)
users: (2000, 13)
merchants (400, 9)
fraud_labels: (20000, 11)


In [4]:
# Step 4 — Inspect columns and data types of all 4 tables

print('=' * 50)
print('TRANSACTIONS')
print('=' * 50)
print(transactions.info())



TRANSACTIONS
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   transaction_id               20000 non-null  str    
 1   user_id                      20000 non-null  str    
 2   receiver_id                  20000 non-null  str    
 3   receiver_type                20000 non-null  str    
 4   amount                       20000 non-null  float64
 5   timestamp                    20000 non-null  str    
 6   date                         20000 non-null  str    
 7   hour_of_day                  20000 non-null  int64  
 8   day_of_week                  20000 non-null  str    
 9   is_weekend                   20000 non-null  int64  
 10  is_night_transaction         20000 non-null  int64  
 11  time_since_last_txn_min      17665 non-null  float64
 12  transaction_type             20000 non-null  str    
 13  payment_app   

In [5]:
print('=' * 50)
print('USERS')
print('=' * 50)
print(users.info())

USERS
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   2000 non-null   str    
 1   age_group                 2000 non-null   str    
 2   city                      2000 non-null   str    
 3   city_tier                 2000 non-null   str    
 4   kyc_status                2000 non-null   str    
 5   account_age_days          2000 non-null   int64  
 6   linked_bank_count         2000 non-null   int64  
 7   avg_monthly_transactions  2000 non-null   int64  
 8   avg_transaction_value     2000 non-null   float64
 9   preferred_app             2000 non-null   str    
 10  preferred_device          2000 non-null   str    
 11  user_loyalty_score        2000 non-null   float64
 12  is_high_risk_user         2000 non-null   int64  
dtypes: float64(2), int64(4), str(7)
memory usage: 203.3 KB
None


In [6]:
print('=' * 50)
print('MERCHANTS')
print('=' * 50)
print(merchants.info())

MERCHANTS
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   merchant_id             400 non-null    str    
 1   merchant_name           400 non-null    str    
 2   merchant_category       400 non-null    str    
 3   merchant_size           400 non-null    str    
 4   city                    400 non-null    str    
 5   city_tier               400 non-null    str    
 6   avg_daily_transactions  400 non-null    int64  
 7   is_registered           400 non-null    int64  
 8   rating                  400 non-null    float64
dtypes: float64(1), int64(2), str(6)
memory usage: 28.3 KB
None


In [7]:
print('=' * 50)
print('FRAUD LABELS')
print('=' * 50)
print(fraud_labels.info())

FRAUD LABELS
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   transaction_id            20000 non-null  str    
 1   user_id                   20000 non-null  str    
 2   receiver_id               20000 non-null  str    
 3   amount                    20000 non-null  float64
 4   timestamp                 20000 non-null  str    
 5   is_fraud                  20000 non-null  int64  
 6   new_device_flag           20000 non-null  int64  
 7   ip_location_mismatch      20000 non-null  int64  
 8   failed_attempts_last_24h  20000 non-null  int64  
 9   transaction_velocity      19607 non-null  float64
 10  amount_deviation_score    19633 non-null  float64
dtypes: float64(3), int64(4), str(4)
memory usage: 1.7 MB
None


In [8]:
# Step 5 — Quick look at first 5 and last 5  rows of each table

transactions.head(5)


,transaction_id,user_id,receiver_id,receiver_type,amount,timestamp,date,hour_of_day,day_of_week,is_weekend,is_night_transaction,time_since_last_txn_min,transaction_type,payment_app,device_type,status,user_city_tier,user_kyc_status,user_avg_monthly_txn,user_avg_txn_value,user_loyalty_score,new_device_flag,ip_location_mismatch,failed_attempts_last_24h,transaction_velocity,amount_deviation_score,is_fraud,recurring_payment_flag,balance_after_transaction,transaction_frequency_score
0,TXN0013171,USR01949,MRC0069,Merchant,1533.88,2024-01-01 00:21:17,2024-01-01,0,Monday,0,1,-30240.60,EMI,Paytm,Android,Success,Tier 2,Verified,36,472.44,0.194,0,0,2,0.0,2.2467,0,1,36023.96,0.72
1,TXN0006480,USR01975,MRC0048,Merchant,1371.57,2024-01-01 00:36:29,2024-01-01,0,Monday,0,1,-351322.38,P2M,Paytm,Web,Success,Tier 2,Verified,34,629.94,0.213,0,0,0,4.0,1.1773,0,0,65138.89,0.68
2,TXN0001430,USR01127,MRC0312,Merchant,500.50,2024-01-01 00:47:46,2024-01-01,0,Monday,0,1,-60756.05,Subscription,Amazon Pay,iOS,Success,Tier 1,Verified,43,436.30,0.381,0,0,0,1.0,0.1471,0,1,69347.11,0.86
3,TXN0003501,USR00827,USR01877,User,724.05,2024-01-01 00:52:07,2024-01-01,0,Monday,0,1,-262998.52,P2P,WhatsApp Pay,Android,Success,Tier 1,Verified,62,278.28,0.238,0,0,1,0.0,1.6019,0,0,52305.43,1.00
4,TXN0004423,USR00673,USR00534,User,128.22,2024-01-01 01:22:31,2024-01-01,1,Monday,0,1,-418835.00,P2P,GPay,iOS,Success,Tier 3,Verified,24,185.17,0.187,0,0,5,1.0,0.3076,0,0,9601.13,0.48


In [9]:
transactions.tail(5)

,transaction_id,user_id,receiver_id,receiver_type,amount,timestamp,date,hour_of_day,day_of_week,is_weekend,is_night_transaction,time_since_last_txn_min,transaction_type,payment_app,device_type,status,user_city_tier,user_kyc_status,user_avg_monthly_txn,user_avg_txn_value,user_loyalty_score,new_device_flag,ip_location_mismatch,failed_attempts_last_24h,transaction_velocity,amount_deviation_score,is_fraud,recurring_payment_flag,balance_after_transaction,transaction_frequency_score
19995,TXN0018270,USR00228,USR01463,User,70.26,2024-12-30 21:59:07,2024-12-30,21,Monday,0,0,98565.25,P2P,GPay,Android,Success,Tier 3,Verified,23,127.18,0.114,0,0,2,1.0,0.4476,1,0,63553.01,0.46
19996,TXN0006448,USR01360,MRC0360,Merchant,414.20,2024-12-30 22:08:44,2024-12-30,22,Monday,0,1,244367.03,EMI,GPay,Web,Success,Tier 1,Not Verified,50,621.82,0.781,1,0,0,1.0,0.3339,0,1,43470.82,1.00
19997,TXN0013837,USR00582,USR00308,User,63.32,2024-12-30 22:09:27,2024-12-30,22,Monday,0,1,59702.07,P2P,GPay,Web,Success,Tier 2,Verified,30,257.45,0.855,0,0,0,0.0,0.7540,0,0,62472.32,0.60
19998,TXN0009361,USR00557,MRC0317,Merchant,193.06,2024-12-30 22:22:16,2024-12-30,22,Monday,0,1,318860.72,Bill Payment,PhonePe,iOS,Success,Tier 2,Verified,25,183.27,0.513,0,0,0,2.0,0.0534,0,1,40969.58,0.50
19999,TXN0017978,USR01600,MRC0120,Merchant,7019.75,2024-12-30 22:31:52,2024-12-30,22,Monday,0,1,169560.10,P2M,PhonePe,iOS,Success,Tier 1,Not Verified,55,1780.47,0.783,0,0,2,0.0,2.9426,0,0,26613.04,1.00


In [10]:
users.head(5)

,user_id,age_group,city,city_tier,kyc_status,account_age_days,linked_bank_count,avg_monthly_transactions,avg_transaction_value,preferred_app,preferred_device,user_loyalty_score,is_high_risk_user
0,USR00001,55+,Surat,Tier 2,Verified,486,1,19,629.65,GPay,Android,0.348,0
1,USR00002,35-44,Tiruchirappalli,Tier 3,Verified,449,2,15,429.28,GPay,Android,0.903,0
2,USR00003,25-34,Pune,Tier 1,Verified,160,1,56,340.67,GPay,Android,0.297,0
3,USR00004,25-34,Jaipur,Tier 2,Verified,2328,1,30,221.18,PhonePe,Android,0.685,0
4,USR00005,18-24,Nagpur,Tier 2,Verified,1869,2,24,918.09,GPay,iOS,0.828,1


In [11]:
users.tail(5)

,user_id,age_group,city,city_tier,kyc_status,account_age_days,linked_bank_count,avg_monthly_transactions,avg_transaction_value,preferred_app,preferred_device,user_loyalty_score,is_high_risk_user
1995,USR01996,45-54,Visakhapatnam,Tier 2,Verified,691,1,35,385.83,Paytm,iOS,0.619,0
1996,USR01997,35-44,Delhi,Tier 1,Verified,763,2,45,613.90,PhonePe,Android,0.515,0
1997,USR01998,35-44,Chennai,Tier 1,Verified,1840,2,39,1464.63,GPay,Web,0.226,0
1998,USR01999,18-24,Solapur,Tier 3,Verified,2029,1,12,177.91,BHIM,iOS,0.873,0
1999,USR02000,35-44,Meerut,Tier 3,Verified,558,1,23,498.19,Amazon Pay,Android,0.561,0


In [12]:
merchants.head(5)

,merchant_id,merchant_name,merchant_category,merchant_size,city,city_tier,avg_daily_transactions,is_registered,rating
0,MRC0001,Merchant_1,Insurance,Medium,Bhopal,Tier 2,60,1,4.2
1,MRC0002,Merchant_2,Travel,Small,Coimbatore,Tier 2,7,1,4.7
2,MRC0003,Merchant_3,Utilities,Small,Mumbai,Tier 1,4,1,4.1
3,MRC0004,Merchant_4,Grocery,Small,Rajkot,Tier 2,16,1,4.2
4,MRC0005,Merchant_5,Travel,Small,Mumbai,Tier 1,10,1,2.8


In [13]:
merchants.tail(5)

,merchant_id,merchant_name,merchant_category,merchant_size,city,city_tier,avg_daily_transactions,is_registered,rating
395,MRC0396,Merchant_396,Utilities,Small,Kolkata,Tier 1,6,1,4.8
396,MRC0397,Merchant_397,Healthcare,Medium,Kochi,Tier 2,45,1,4.8
397,MRC0398,Merchant_398,Entertainment,Medium,Indore,Tier 2,49,1,4.4
398,MRC0399,Merchant_399,Recharge,Medium,Hyderabad,Tier 1,48,1,3.4
399,MRC0400,Merchant_400,Food & Dining,Small,Chandigarh,Tier 2,12,1,4.7


In [14]:
fraud_labels.head(5)

,transaction_id,user_id,receiver_id,amount,timestamp,is_fraud,new_device_flag,ip_location_mismatch,failed_attempts_last_24h,transaction_velocity,amount_deviation_score
0,TXN0000001,USR01603,MRC0374,1650.84,2024-12-26 14:52:00,0,0,0,0,0.0,0.5146
1,TXN0000002,USR01759,MRC0274,599.20,2024-06-29 00:09:24,0,0,0,1,0.0,0.7026
2,TXN0000003,USR01894,MRC0122,261.36,2024-02-21 05:08:33,0,0,0,0,0.0,0.1547
3,TXN0000004,USR00606,USR00692,215.71,2024-08-09 18:52:33,0,0,0,0,0.0,0.0078
4,TXN0000005,USR01246,MRC0007,411.15,2024-02-18 06:37:11,0,0,0,1,1.0,0.1170


In [15]:
fraud_labels.tail(5)

,transaction_id,user_id,receiver_id,amount,timestamp,is_fraud,new_device_flag,ip_location_mismatch,failed_attempts_last_24h,transaction_velocity,amount_deviation_score
19995,TXN0019996,USR00568,MRC0014,231.70,2024-02-29 20:41:15,0,0,0,1,0.0,0.1763
19996,TXN0019997,USR01644,MRC0031,587.95,2024-04-22 14:29:27,0,1,0,1,1.0,0.1640
19997,TXN0019998,USR00113,MRC0014,1126.20,2024-11-21 15:00:16,0,0,0,0,0.0,0.9166
19998,TXN0019999,USR01565,MRC0188,436.98,2024-11-14 17:25:08,0,0,0,0,0.0,NaN
19999,TXN0020000,USR00525,MRC0048,463.66,2024-06-17 05:36:46,0,0,1,0,1.0,0.4272


In [16]:
# Step 6 — Check for duplicate rows in all 4 tables

print(f'Duplicated rows in transactions: {transactions.duplicated().sum()}')
print(f'Duplicated rows in users: {users.duplicated().sum()}')
print(f'Duplicated rows in merchants: {merchants.duplicated().sum()}')
print(f'Duplicated rows in fraud labels: {fraud_labels.duplicated().sum()}')

Duplicated rows in transactions: 0
Duplicated rows in users: 0
Duplicated rows in merchants: 0
Duplicated rows in fraud labels: 0


## Phase 1 — Initial Inspection Summary

| Table | Rows | Columns | Nulls Found | Data Type Issues |
|---|---|---|---|---|
| transactions | 20,000 | 30 | 3 columns | timestamp, date as string |
| users | 2,000 | 13 | None | None |
| merchants | 400 | 9 | None | None |
| fraud_labels | 20,000 | 11 | 2 columns | timestamp as string |

### Issues to Fix in Phase 2 — Data Cleaning
1. Fix timestamp and date columns from string to datetime in transactions
2. Fix timestamp column from string to datetime in fraud_labels
3. Handle nulls in time_since_last_txn_min — 2,335 missing values
4. Handle nulls in transaction_velocity — 393 missing values
5. Handle nulls in amount_deviation_score — 367 missing values

## Phase 2 — Data Cleaning & Preprocessing

### What is Data Cleaning?
Data Cleaning is the process of identifying and correcting errors, 
inconsistencies, and inaccuracies in raw data before analysis. 
Real-world data is never perfect — even well-structured fintech 
data has missing values, wrong data types, and inconsistencies 
that must be fixed before loading into a database.

### Why Data Cleaning Matters in Fintech
In a real payment system, dirty data leads to wrong analysis. 
If timestamp is stored as a string, you cannot do time-based 
fraud detection. If null values in transaction_velocity are not 
handled, your fraud scoring queries will return incorrect results. 
Clean data is the foundation of trustworthy analysis.

### What We Are Fixing in This Phase
1. Convert timestamp and date columns to proper datetime format
2. Handle null values in 3 columns using appropriate strategies
3. Validate relational keys between all 4 tables
4. Export clean CSVs to data/clean folder for PostgreSQL loading

In [19]:
for col in transactions.columns:
    print(f"{col}: {transactions[col].dtype} | Null: {transactions[col].isnull().sum()}")

transaction_id: str | Null: 0
user_id: str | Null: 0
receiver_id: str | Null: 0
receiver_type: str | Null: 0
amount: float64 | Null: 0
timestamp: str | Null: 0
date: str | Null: 0
hour_of_day: int64 | Null: 0
day_of_week: str | Null: 0
is_weekend: int64 | Null: 0
is_night_transaction: int64 | Null: 0
time_since_last_txn_min: float64 | Null: 2335
transaction_type: str | Null: 0
payment_app: str | Null: 0
device_type: str | Null: 0
status: str | Null: 0
user_city_tier: str | Null: 0
user_kyc_status: str | Null: 0
user_avg_monthly_txn: int64 | Null: 0
user_avg_txn_value: float64 | Null: 0
user_loyalty_score: float64 | Null: 0
new_device_flag: int64 | Null: 0
ip_location_mismatch: int64 | Null: 0
failed_attempts_last_24h: int64 | Null: 0
transaction_velocity: float64 | Null: 393
amount_deviation_score: float64 | Null: 367
is_fraud: int64 | Null: 0
recurring_payment_flag: int64 | Null: 0
balance_after_transaction: float64 | Null: 0
transaction_frequency_score: float64 | Null: 0


In [ ]:
#---------------------------------------------------------------
# PHASE 2 — DATA CLEANING & PREPROCESSING
#---------------------------------------------------------------

# Step 1 — Fix datetime columns in transactions and fraud_labels
# timestamp and date are stored as strings — convert to proper datetime format

transactions['date'] = pd.to_datetime(transactions['date'])
transactions['timestamp'] = pd.to_datetime(transactions['timestamp'])

fraud_labels['timestamp'] = pd.to_datetime(fraud_labels['timestamp'])

print(f"transactions timestamp dtype: {transactions['timestamp'].dtype}")
print(f"transactions date dtype: {transactions['date'].dtype}")
print(f"fraud_labels timestamp dtype: {fraud_labels['timestamp'].dtype}")

transactions timestamp dtype: datetime64[us]
transactions date dtype: datetime64[us]
fraud_labels timestamp dtype: datetime64[us]


In [24]:
# Step 2 — Handle null values in transactions table

# time_since_last_txn_min — 2335 nulls
# These are likely first transactions by a user — no previous transaction to calculate time from
# Strategy: fill with median — robust to outliers

median_time = transactions['time_since_last_txn_min'].median()
transactions['time_since_last_txn_min'] = transactions['time_since_last_txn_min'].fillna(median_time)
print(f"time_since_last_txn_min — filled {2335} nulls with median: {median_time:.2f} minutes")

# transaction_velocity — 393 nulls
# Velocity = how many transactions in short window. Null means no rapid activity detected
# Strategy: fill with 0 — no velocity is the safest assumption

transactions['transaction_velocity'] = transactions['transaction_velocity'].fillna(0)
fraud_labels['transaction_velocity'] = fraud_labels['transaction_velocity'].fillna(0)
print(f"transaction_velocity — filled nulls with 0 in transactions and fraud_labels")


# amount_deviation_score — 367 nulls
# Deviation from user's average. Null means insufficient history to calculate
# Strategy: fill with median — represents average deviation behavior
median_deviation = transactions['amount_deviation_score'].median()
transactions['amount_deviation_score'] = transactions['amount_deviation_score'].fillna(median_deviation)
fraud_labels['amount_deviation_score'] = fraud_labels['amount_deviation_score'].fillna(median_deviation)
print(f"amount_deviation_score — filled nulls with median: {median_deviation:.4f}")


print("\n All null values handled successfully")


time_since_last_txn_min — filled 2335 nulls with median: -1724.13 minutes
transaction_velocity — filled nulls with 0 in transactions and fraud_labels
amount_deviation_score — filled nulls with median: 0.4445

 All null values handled successfully


In [25]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   transaction_id               20000 non-null  str           
 1   user_id                      20000 non-null  str           
 2   receiver_id                  20000 non-null  str           
 3   receiver_type                20000 non-null  str           
 4   amount                       20000 non-null  float64       
 5   timestamp                    20000 non-null  datetime64[us]
 6   date                         20000 non-null  datetime64[us]
 7   hour_of_day                  20000 non-null  int64         
 8   day_of_week                  20000 non-null  str           
 9   is_weekend                   20000 non-null  int64         
 10  is_night_transaction         20000 non-null  int64         
 11  time_since_last_txn_min      20000 non-null  float64

In [26]:
fraud_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   transaction_id            20000 non-null  str           
 1   user_id                   20000 non-null  str           
 2   receiver_id               20000 non-null  str           
 3   amount                    20000 non-null  float64       
 4   timestamp                 20000 non-null  datetime64[us]
 5   is_fraud                  20000 non-null  int64         
 6   new_device_flag           20000 non-null  int64         
 7   ip_location_mismatch      20000 non-null  int64         
 8   failed_attempts_last_24h  20000 non-null  int64         
 9   transaction_velocity      20000 non-null  float64       
 10  amount_deviation_score    20000 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(4), str(3)
memory usage: 1.7 MB


In [27]:
# Investigate time_since_last_txn_min values
print(transactions_raw['time_since_last_txn_min'].describe())
print(f"\nNegative values count: {(transactions_raw['time_since_last_txn_min'] < 0).sum()}")

count     17665.000000
mean       -899.133724
std      214733.190266
min     -517802.170000
25%     -154683.080000
50%       -1724.130000
75%      153152.770000
max      524477.870000
Name: time_since_last_txn_min, dtype: float64

Negative values count: 8877


In [28]:
# Step 3 — Verify null counts after cleaning

print("Null counts after cleaning:")
print("\ntransactions:")
print(transactions.isnull().sum()[transactions.isnull().sum() > 0])

print("\nfraud_labels:")
print(fraud_labels.isnull().sum()[fraud_labels.isnull().sum() > 0])

print("\n✅ Null check complete — all nulls handled")

Null counts after cleaning:

transactions:
Series([], dtype: int64)

fraud_labels:
Series([], dtype: int64)

✅ Null check complete — all nulls handled


In [31]:
# Step 4 — Validate relational keys between all 4 tables

# Check 1 — All user_ids in transactions exist in users table
txn_user = set(transactions['user_id'].unique())
users_id = set(users['user_id'].unique())
unmatched_users = txn_user - users_id
print(f'User ids in transaction are not found in users table: {len(unmatched_users)}')

# Check 2 — All transaction_ids in fraud_labels exist in transactions
txn_ids = set(transactions['transaction_id'].unique())
fraud_ids = set(fraud_labels['transaction_id'].unique())
unmatched_fraud = txn_ids - fraud_ids
print(f'Transaction ids in transactions are not found in fraud table: {len(unmatched_fraud)}')

# Check 3 — Merchant receiver_ids in transactions exist in merchants table
merchant_txn = transactions[transactions['receiver_type'] == 'Merchant']['receiver_id'].unique()
merchant_ids = set(merchants['merchant_id'].unique())
unmatched_txn_ids = set(merchant_txn) - merchant_ids
print(f"Merchant IDs in transactions not found in merchants table: {len(unmatched_txn_ids)}")

print("\n Relational key validation complete")


User ids in transaction are not found in users table: 0
Transaction ids in transactions are not found in fraud table: 0
Merchant IDs in transactions not found in merchants table: 0

 Relational key validation complete


In [32]:
# Step 5 — Export clean CSVs to data/clean folder

transactions.to_csv('../data/clean/transactions_clean.csv', index=False)
users.to_csv('../data/clean/users_clean.csv', index=False)
merchants.to_csv('../data/clean/merchants_clean.csv', index=False)
fraud_labels.to_csv('../data/clean/fraud_labels_clean.csv', index=False)

print("✅ All 4 clean CSVs exported to data/clean folder")
print(f"\ntransactions_clean: {transactions.shape}")
print(f"users_clean: {users.shape}")
print(f"merchants_clean: {merchants.shape}")
print(f"fraud_labels_clean: {fraud_labels.shape}")

✅ All 4 clean CSVs exported to data/clean folder

transactions_clean: (20000, 30)
users_clean: (2000, 13)
merchants_clean: (400, 9)
fraud_labels_clean: (20000, 11)


## Phase 2 — Data Cleaning Summary

| Step | Action | Result |
|---|---|---|
| Datetime Fix | Converted timestamp and date from string to datetime | 3 columns fixed across 2 tables |
| Null Handling | time_since_last_txn_min filled with median | 2,335 nulls resolved |
| Null Handling | transaction_velocity filled with 0 | 393 nulls resolved in transactions + fraud_labels |
| Null Handling | amount_deviation_score filled with median | 367 nulls resolved in transactions + fraud_labels |
| Key Validation | Checked foreign keys across all 4 tables | Zero orphaned records found |
| Export | Clean CSVs saved to data/clean folder | 4 files ready for PostgreSQL |

### Final Clean Dataset
- transactions_clean — 20,000 rows × 30 columns
- users_clean — 2,000 rows × 13 columns
- merchants_clean — 400 rows × 9 columns
- fraud_labels_clean — 20,000 rows × 11 columns

### Data is now ready for PostgreSQL loading in Phase 3

## Phase 3 — PostgreSQL Database Setup

### What is happening in this phase?
All 4 clean DataFrames are loaded into PostgreSQL database — 
`upi_intelligence` — using SQLAlchemy. This converts our flat 
CSV files into a proper relational database with 4 connected tables.

### Why PostgreSQL?
PostgreSQL is a production-grade relational database used by companies 
like Razorpay, PhonePe, and Swiggy in their actual data infrastructure. 
Loading our data into PostgreSQL means we can now write complex multi-table 
JOIN queries, CTEs, and window functions.

### Database Setup
- Database: upi_intelligence
- Tables created: transactions, users, merchants, fraud_labels
- Setup script: sql/db_setup.py
- All 4 tables connected via relational keys — user_id and receiver_id